In [1]:
# ============================================================
# MARKETPLACE REVIEW INTELLIGENCE
# Notebook 05 — Modelagem Dimensional e Exportação
# ============================================================
# Objetivo: Transformar o dataset processado em um Star Schema
# com tabelas fato e dimensão prontas para o Power BI.
#
# Modelo gerado:
# ┌─────────────────────────────────────────────┐
# │           fFact_Reviews (central)           │
# │  review_id | produto_id | data_id |         │
# │  sentimento_id | rating | score |           │
# │  qtd_palavras | review_curto                │
# └──────┬──────────────┬──────────────┬────────┘
#        │              │              │
#   dDim_Produto   dDim_Data   dDim_Sentimento
# ============================================================

import pandas as pd
from pathlib import Path

# Caminhos do projeto
ROOT        = Path().resolve().parent
PROCESSED   = ROOT / "data" / "processed"
OUTPUT      = ROOT / "data" / "output"

# Garante que a pasta output existe
OUTPUT.mkdir(parents=True, exist_ok=True)

# Carrega o dataset final do Notebook 04
df = pd.read_csv(
    PROCESSED / "reviews_nlp.csv",
    encoding="utf-8-sig"
)
df["date"] = pd.to_datetime(df["date"])

print(f"✅ Dataset carregado: {df.shape[0]:,} registros")
print(f"📋 Colunas disponíveis: {list(df.columns)}")

✅ Dataset carregado: 202,785 registros
📋 Colunas disponíveis: ['date', 'rating', 'content', 'product_url', 'qtd_palavras', 'review_curto', 'content_original', 'produto_id', 'slug', 'nome_produto', 'categoria', 'sentimento_score', 'classificacao_sentimento']


In [2]:
# ============================================================
# dDim_Data — Dimensão de Tempo
# ============================================================
# Expande a coluna date em múltiplas colunas temporais
# Essencial para análises de tendência no Power BI
# ============================================================

# Mapeamentos para PT-BR
NOMES_MESES = {
    1: "Janeiro", 2: "Fevereiro", 3: "Março",
    4: "Abril",   5: "Maio",      6: "Junho",
    7: "Julho",   8: "Agosto",    9: "Setembro",
    10: "Outubro", 11: "Novembro", 12: "Dezembro"
}

NOMES_DIAS = {
    0: "Segunda-feira", 1: "Terça-feira",
    2: "Quarta-feira",  3: "Quinta-feira",
    4: "Sexta-feira",   5: "Sábado",
    6: "Domingo"
}

# Cria a dimensão de data com todas as datas únicas
datas_unicas = df["date"].drop_duplicates().sort_values().reset_index(drop=True)

dDim_Data = pd.DataFrame()
dDim_Data["data_id"]     = range(1, len(datas_unicas) + 1)
dDim_Data["data"]        = datas_unicas.values
dDim_Data["dia"]         = dDim_Data["data"].dt.day
dDim_Data["mes"]         = dDim_Data["data"].dt.month
dDim_Data["ano"]         = dDim_Data["data"].dt.year
dDim_Data["trimestre"]   = dDim_Data["data"].dt.quarter
dDim_Data["semestre"]    = dDim_Data["mes"].apply(lambda m: 1 if m <= 6 else 2)
dDim_Data["nome_mes"]    = dDim_Data["mes"].map(NOMES_MESES)
dDim_Data["dia_semana"]  = dDim_Data["data"].dt.dayofweek
dDim_Data["nome_dia"]    = dDim_Data["dia_semana"].map(NOMES_DIAS)
dDim_Data["ano_mes"]     = dDim_Data["data"].dt.to_period("M").astype(str)
dDim_Data["data"]        = dDim_Data["data"].dt.date

print(f"✅ dDim_Data criada: {dDim_Data.shape[0]:,} linhas × {dDim_Data.shape[1]} colunas")
print(f"\n📅 Range temporal:")
print(f"   De: {dDim_Data['data'].min()}")
print(f"   Até: {dDim_Data['data'].max()}")
print(f"\nAmostra:")
display(dDim_Data.head(5))

✅ dDim_Data criada: 2,119 linhas × 11 colunas

📅 Range temporal:
   De: 2015-11-01
   Até: 2025-02-19

Amostra:


,data_id,data,dia,mes,ano,trimestre,semestre,nome_mes,dia_semana,nome_dia,ano_mes
0,1,2015-11-01,1,11,2015,4,2,Novembro,6,Domingo,2015-11
1,2,2016-05-05,5,5,2016,2,1,Maio,3,Quinta-feira,2016-05
2,3,2016-07-19,19,7,2016,3,2,Julho,1,Terça-feira,2016-07
3,4,2016-10-03,3,10,2016,4,2,Outubro,0,Segunda-feira,2016-10
4,5,2016-10-23,23,10,2016,4,2,Outubro,6,Domingo,2016-10


In [3]:
# ============================================================
# dDim_Produto — Dimensão de Produto
# ============================================================
# Uma linha por produto único
# Métricas (rating_medio, total_reviews) serão calculadas
# via DAX no Power BI — boas práticas de modelagem dimensional
# ============================================================

dDim_Produto = (
    df[["produto_id", "nome_produto", "categoria", "product_url"]]
    .drop_duplicates(subset=["produto_id"])
    .reset_index(drop=True)
)

# Garante que não há produto_id nulo
dDim_Produto = dDim_Produto[dDim_Produto["produto_id"].notna()]

# Ordena por produto_id para consistência
dDim_Produto = dDim_Produto.sort_values("produto_id").reset_index(drop=True)

print(f"✅ dDim_Produto criada: {dDim_Produto.shape[0]:,} linhas × {dDim_Produto.shape[1]} colunas")
print(f"\n📦 Distribuição por categoria:")
dist = dDim_Produto["categoria"].value_counts()
pct  = (dist / len(dDim_Produto) * 100).round(2)
display(pd.DataFrame({"Produtos": dist, "% do Total": pct}))
print(f"\nAmostra:")
display(dDim_Produto.head(5))

✅ dDim_Produto criada: 17,972 linhas × 4 colunas

📦 Distribuição por categoria:


,Produtos,% do Total
categoria,,
Kit Capilar,3696,20.57
Escova Progressiva,3673,20.44
Máscara Capilar,1267,7.05
Óleo Capilar,1052,5.85
Antiqueda / Tônico,1036,5.76
Outros Cosméticos,1024,5.70
Sérum / Leave-in,974,5.42
Shampoo,939,5.22
Tintura / Matizador,895,4.98



Amostra:


,produto_id,nome_produto,categoria,product_url
0,MLB-1000227287,Kit Ruiva Cobre Effect Amend Flamingo Kamaleo ...,Kit Capilar,https://produto.mercadolivre.com.br/MLB-100022...
1,MLB-1001626745,24 Reparador De Pontas Semi De Lino Anjore Ata...,Sérum / Leave-in,https://produto.mercadolivre.com.br/MLB-100162...
2,MLB-1001669434,Truss Amino,Outros Cosméticos,https://produto.mercadolivre.com.br/MLB-100166...
3,MLB-1002668883,Kit 2 Escovas Progressivas Eternity Liss Perola,Escova Progressiva,https://produto.mercadolivre.com.br/MLB-100266...
4,MLB-1003535687,Como Fazer O Cabelo Crescer Mais Rapido Produt...,Outros Cosméticos,https://produto.mercadolivre.com.br/MLB-100353...


In [4]:
# ============================================================
# dDim_Sentimento — Dimensão de Sentimento
# ============================================================
# Tabela de lookup para classificações de sentimento
# Inclui metadados úteis para filtros no Power BI
# ============================================================

dDim_Sentimento = pd.DataFrame({
    "sentimento_id": [1, 2, 3],

    "classificacao": ["Positivo", "Neutro", "Negativo"],

    "faixa_score": [
        "score > 0.05",
        "score entre -0.05 e 0.05",
        "score < -0.05"
    ],

    "descricao": [
        "Cliente satisfeito com o produto",
        "Review sem polaridade clara",
        "Cliente insatisfeito com o produto"
    ],

    "ordem_exibicao": [1, 2, 3],

    # Cor sugerida para o dashboard
    "cor_hex": ["#2ECC71", "#F39C12", "#E74C3C"]
})

print(f"✅ dDim_Sentimento criada: {dDim_Sentimento.shape[0]} linhas × {dDim_Sentimento.shape[1]} colunas")
display(dDim_Sentimento)

✅ dDim_Sentimento criada: 3 linhas × 6 colunas


,sentimento_id,classificacao,faixa_score,descricao,ordem_exibicao,cor_hex
0,1,Positivo,score > 0.05,Cliente satisfeito com o produto,1,#2ECC71
1,2,Neutro,score entre -0.05 e 0.05,Review sem polaridade clara,2,#F39C12
2,3,Negativo,score < -0.05,Cliente insatisfeito com o produto,3,#E74C3C


In [5]:
# ============================================================
# fFact_Reviews — Tabela Fato Central
# ============================================================
# Uma linha por review — conecta todas as dimensões
# Contém apenas chaves estrangeiras e métricas numéricas
# ============================================================

# Cria mapa de data → data_id
mapa_data = dict(zip(
    pd.to_datetime(dDim_Data["data"]),
    dDim_Data["data_id"]
))

# Cria mapa de classificacao → sentimento_id
mapa_sentimento = dict(zip(
    dDim_Sentimento["classificacao"],
    dDim_Sentimento["sentimento_id"]
))

# Monta a tabela fato
fFact_Reviews = pd.DataFrame()

# Chave primária
fFact_Reviews["review_id"] = range(1, len(df) + 1)

# Chaves estrangeiras
fFact_Reviews["produto_id"]    = df["produto_id"].values
fFact_Reviews["data_id"]       = df["date"].map(mapa_data).values
fFact_Reviews["sentimento_id"] = df["classificacao_sentimento"].map(mapa_sentimento).values

# Métricas
fFact_Reviews["rating"]            = df["rating"].values
fFact_Reviews["sentimento_score"]  = df["sentimento_score"].values
fFact_Reviews["qtd_palavras"]      = df["qtd_palavras"].values
fFact_Reviews["review_curto"]      = df["review_curto"].astype(int).values

# Campo de texto — mantido na fato para tooltips no Power BI
fFact_Reviews["content"]           = df["content"].values

print(f"✅ fFact_Reviews criada: {fFact_Reviews.shape[0]:,} linhas × {fFact_Reviews.shape[1]} colunas")
print(f"\nAmostra:")
display(fFact_Reviews.head(5))
print(f"\n📋 Tipos de dados:")
print(fFact_Reviews.dtypes)

✅ fFact_Reviews criada: 202,785 linhas × 9 colunas

Amostra:


,review_id,produto_id,data_id,sentimento_id,rating,sentimento_score,qtd_palavras,review_curto,content
0,1,MLB-3149572356,1590,1,5,1.0,1,1,top.
1,2,MLB-3149572356,1935,1,5,1.0,6,0,"produto bom, cumpre o que promete."
2,3,MLB-3149572356,2115,1,5,1.0,2,1,ótima qualidade.
3,4,MLB-3149572356,2111,1,4,1.0,1,1,bom.
4,5,MLB-3149572356,2079,1,5,1.0,3,0,atendeu minhas expectativas.



📋 Tipos de dados:
review_id             int64
produto_id           object
data_id               int64
sentimento_id         int64
rating                int64
sentimento_score    float64
qtd_palavras          int64
review_curto          int64
content              object
dtype: object


In [6]:
fFact_Reviews.tail(15)

,review_id,produto_id,data_id,sentimento_id,rating,sentimento_score,qtd_palavras,review_curto,content
202770,202771,MLB19567040,2024,1,3,1.0000,3,0,já tive melhor.
202771,202772,MLB19567040,1978,3,1,0.0000,9,0,"não vale a pena, pelo valor e custo beneficio."
202772,202773,MLB19567040,1923,3,3,-1.0000,3,0,arde no rosto.
202773,202774,MLB19567040,1873,2,3,0.0000,13,0,"não achei lá gde coisa, achei caro. acho que n..."
202774,202775,MLB19567040,1789,2,3,0.0000,20,0,dercos mil vezes melhor esse shampoo deixa res...
202775,202776,MLB19567040,1379,3,2,0.0000,13,0,"para meu cabelo não deu certo ,pq já é seco fi..."
202776,202777,MLB19567040,1469,3,2,0.0000,5,0,não resolve nada para oleosidade.
202777,202778,MLB19506399,2047,1,5,1.0000,2,1,excelente produto.
202778,202779,MLB19506399,2008,1,5,1.0000,2,1,produto excelente.
202779,202780,MLB19506399,1665,1,5,1.0000,2,1,super bom.


In [7]:
# ============================================================
# Validação de integridade referencial
# Garante que todas as chaves da fato existem nas dimensões
# ============================================================

print("=" * 55)
print("VALIDAÇÃO DO STAR SCHEMA")
print("=" * 55)

# --- Produto ---
ids_fato     = set(fFact_Reviews["produto_id"].dropna())
ids_dim      = set(dDim_Produto["produto_id"])
orfaos_prod  = ids_fato - ids_dim

print(f"\n📦 PRODUTO")
print(f"   IDs na fato:       {len(ids_fato):,}")
print(f"   IDs na dimensão:   {len(ids_dim):,}")
print(f"   Órfãos (sem dim):  {len(orfaos_prod):,}")
if len(orfaos_prod) == 0:
    print(f"   ✅ Integridade referencial OK")
else:
    print(f"   ⚠️  {len(orfaos_prod)} IDs sem correspondência na dimensão")

# --- Data ---
ids_data_fato = set(fFact_Reviews["data_id"].dropna())
ids_data_dim  = set(dDim_Data["data_id"])
orfaos_data   = ids_data_fato - ids_data_dim

print(f"\n📅 DATA")
print(f"   IDs na fato:       {len(ids_data_fato):,}")
print(f"   IDs na dimensão:   {len(ids_data_dim):,}")
print(f"   Órfãos (sem dim):  {len(orfaos_data):,}")
if len(orfaos_data) == 0:
    print(f"   ✅ Integridade referencial OK")
else:
    print(f"   ⚠️  {len(orfaos_data)} IDs sem correspondência na dimensão")

# --- Sentimento ---
ids_sent_fato = set(fFact_Reviews["sentimento_id"].dropna())
ids_sent_dim  = set(dDim_Sentimento["sentimento_id"])
orfaos_sent   = ids_sent_fato - ids_sent_dim

print(f"\n💬 SENTIMENTO")
print(f"   IDs na fato:       {len(ids_sent_fato):,}")
print(f"   IDs na dimensão:   {len(ids_sent_dim):,}")
print(f"   Órfãos (sem dim):  {len(orfaos_sent):,}")
if len(orfaos_sent) == 0:
    print(f"   ✅ Integridade referencial OK")
else:
    print(f"   ⚠️  {len(orfaos_sent)} IDs sem correspondência na dimensão")

# --- Resumo final ---
print(f"\n{'=' * 55}")
print(f"RESUMO DO STAR SCHEMA")
print(f"{'=' * 55}")
print(f"   fFact_Reviews   → {fFact_Reviews.shape[0]:,} linhas × {fFact_Reviews.shape[1]} colunas")
print(f"   dDim_Produto    → {dDim_Produto.shape[0]:,} linhas × {dDim_Produto.shape[1]} colunas")
print(f"   dDim_Data       → {dDim_Data.shape[0]:,} linhas × {dDim_Data.shape[1]} colunas")
print(f"   dDim_Sentimento → {dDim_Sentimento.shape[0]} linhas × {dDim_Sentimento.shape[1]} colunas")

total_orfaos = len(orfaos_prod) + len(orfaos_data) + len(orfaos_sent)
if total_orfaos == 0:
    print(f"\n✅ Star Schema validado — integridade referencial 100%")
else:
    print(f"\n⚠️  {total_orfaos} registros órfãos encontrados — revisar antes de exportar")

VALIDAÇÃO DO STAR SCHEMA

📦 PRODUTO
   IDs na fato:       17,972
   IDs na dimensão:   17,972
   Órfãos (sem dim):  0
   ✅ Integridade referencial OK

📅 DATA
   IDs na fato:       2,119
   IDs na dimensão:   2,119
   Órfãos (sem dim):  0
   ✅ Integridade referencial OK

💬 SENTIMENTO
   IDs na fato:       3
   IDs na dimensão:   3
   Órfãos (sem dim):  0
   ✅ Integridade referencial OK

RESUMO DO STAR SCHEMA
   fFact_Reviews   → 202,785 linhas × 9 colunas
   dDim_Produto    → 17,972 linhas × 4 colunas
   dDim_Data       → 2,119 linhas × 11 colunas
   dDim_Sentimento → 3 linhas × 6 colunas

✅ Star Schema validado — integridade referencial 100%


In [8]:
# ============================================================
# Exporta as 4 tabelas em CSV para o Power BI
# Nomenclatura com prefixo fato_ e dim_ documenta o modelo
# ============================================================

tabelas = {
    "fato_reviews":     fFact_Reviews,
    "dim_produto":      dDim_Produto,
    "dim_data":         dDim_Data,
    "dim_sentimento":   dDim_Sentimento,
}

print("=" * 55)
print("EXPORTAÇÃO DAS TABELAS")
print("=" * 55)

for nome, tabela in tabelas.items():
    caminho = OUTPUT / f"{nome}.csv"
    tabela.to_csv(caminho, index=False, encoding="utf-8-sig")
    print(f"\n✅ {nome}.csv exportado")
    print(f"   📁 {caminho}")
    print(f"   📊 {tabela.shape[0]:,} linhas × {tabela.shape[1]} colunas")
    print(f"   📋 Colunas: {list(tabela.columns)}")

print(f"\n{'=' * 55}")
print(f"🎉 Pipeline ETL concluído com sucesso!")
print(f"{'=' * 55}")
print(f"\n📁 Arquivos prontos em: {OUTPUT}")
print(f"\nPróximo passo → importar os 4 CSVs no Power BI")
print(f"e construir os relacionamentos do Star Schema.")

EXPORTAÇÃO DAS TABELAS

✅ fato_reviews.csv exportado
   📁 D:\GITHUB\portfolio-analista-dados\projetos\marketplace-review-intelligence\data\output\fato_reviews.csv
   📊 202,785 linhas × 9 colunas
   📋 Colunas: ['review_id', 'produto_id', 'data_id', 'sentimento_id', 'rating', 'sentimento_score', 'qtd_palavras', 'review_curto', 'content']

✅ dim_produto.csv exportado
   📁 D:\GITHUB\portfolio-analista-dados\projetos\marketplace-review-intelligence\data\output\dim_produto.csv
   📊 17,972 linhas × 4 colunas
   📋 Colunas: ['produto_id', 'nome_produto', 'categoria', 'product_url']

✅ dim_data.csv exportado
   📁 D:\GITHUB\portfolio-analista-dados\projetos\marketplace-review-intelligence\data\output\dim_data.csv
   📊 2,119 linhas × 11 colunas
   📋 Colunas: ['data_id', 'data', 'dia', 'mes', 'ano', 'trimestre', 'semestre', 'nome_mes', 'dia_semana', 'nome_dia', 'ano_mes']

✅ dim_sentimento.csv exportado
   📁 D:\GITHUB\portfolio-analista-dados\projetos\marketplace-review-intelligence\data\output\dim